# Exploring `reserve_subnet_owner_share`

Interactive walk-through of the helper that reserves `SUBNET_OWNER_WEIGHT_SHARE` of every weight submission for the subnet owner (uid=0). Each section feeds a realistic `{uid: weight}` dict in and inspects what comes out — both straight from the helper and after the downstream `_normalize_uid_weights` step that actually runs before `set_weights`.

All numerical checks below derive from the `SUBNET_OWNER_WEIGHT_SHARE` constant rather than hardcoding a literal, so the notebook tracks the policy if the share is changed.

Run from the repo root so `connito.*` imports resolve.

In [ ]:
from __future__ import annotations

import math

from connito.shared.chain import (
    SUBNET_OWNER_UID,
    SUBNET_OWNER_WEIGHT_SHARE,
    _normalize_uid_weights,
    reserve_subnet_owner_share,
)

print(f"SUBNET_OWNER_UID          = {SUBNET_OWNER_UID}")
print(f"SUBNET_OWNER_WEIGHT_SHARE = {SUBNET_OWNER_WEIGHT_SHARE}")

In [ ]:
def show(label, weights):
    """Pretty-print a `{uid: weight}` dict sorted desc by weight."""
    total = sum(weights.values())
    print(f"  {label}  (n={len(weights)}, sum={total:.6f})")
    for uid, w in sorted(weights.items(), key=lambda kv: -kv[1]):
        marker = "  <-- owner" if uid == SUBNET_OWNER_UID else ""
        print(f"    uid={uid:>4}  w={w:.6f}  ({100*w/max(total,1e-12):5.2f}%)" + marker)


def assert_close(actual, expected, tol=1e-9, msg=""):
    if not math.isclose(actual, expected, abs_tol=tol):
        raise AssertionError(f"{msg}: got {actual}, expected {expected}")
    print(f"  [PASS] {msg}: {actual:.6f} ~= {expected:.6f}")

## 1. Basic case

Two miners, equal weight. Helper should scale both by `(1 - SUBNET_OWNER_WEIGHT_SHARE)` and add uid=0 at `SUBNET_OWNER_WEIGHT_SHARE`. Final sum stays at 1.0.

In [ ]:
inp = {1: 0.5, 2: 0.5}
out = reserve_subnet_owner_share(inp)

show("input ", inp)
show("output", out)

scale = 1 - SUBNET_OWNER_WEIGHT_SHARE
assert_close(out[1], 0.5 * scale, msg="miner-1 scaled by (1 - share)")
assert_close(out[2], 0.5 * scale, msg="miner-2 scaled by (1 - share)")
assert_close(out[SUBNET_OWNER_UID], SUBNET_OWNER_WEIGHT_SHARE, msg="owner share = SUBNET_OWNER_WEIGHT_SHARE")
assert_close(sum(out.values()), 1.0, msg="output sums to 1.0")

## 2. Realistic cohort-style payload

What the validator actually produces in a normal round: a few G1 winners sharing 98%, a few G2 entries sharing 2%, weights inside each group split in proportion to local score. uid=0 should end up between G1 and G2 in size after the reserve.

In [ ]:
# Mimics what `compute_uid_weights` produces with G1_share=0.98, G2_share=0.02.
g1_uids = {12: 0.45, 27: 0.33, 41: 0.20}  # in-group split by score, summing to 0.98
g2_uids = {88: 0.012, 91: 0.008}           # in-group split by score, summing to 0.02

# Scale to the right shares.
g1 = {u: w / sum(g1_uids.values()) * 0.98 for u, w in g1_uids.items()}
g2 = {u: w / sum(g2_uids.values()) * 0.02 for u, w in g2_uids.items()}
inp = {**g1, **g2}

out = reserve_subnet_owner_share(inp)

show("input ", inp)
show("output", out)

scale = 1 - SUBNET_OWNER_WEIGHT_SHARE
assert_close(sum(out.values()), 1.0, msg="sum preserved")
assert_close(out[SUBNET_OWNER_UID], SUBNET_OWNER_WEIGHT_SHARE, msg="owner gets exactly SUBNET_OWNER_WEIGHT_SHARE")

# Ratios among the original miners are preserved.
for u in inp:
    assert_close(out[u] / inp[u], scale, msg=f"uid={u} scaled by (1 - share)")

## 3. Edge cases

Each one short-circuits a different branch of the helper.

In [ ]:
# Empty input -> owner takes all of it.
print("empty input")
out = reserve_subnet_owner_share({})
show("output", out)
assert out == {SUBNET_OWNER_UID: 1.0}
print("  [PASS] empty -> {0: 1.0}")

In [ ]:
# Owner already in input -> their existing weight is scaled, then SUBNET_OWNER_WEIGHT_SHARE added on top.
print("owner already in input")
inp = {0: 0.2, 1: 0.8}
out = reserve_subnet_owner_share(inp)
show("input ", inp)
show("output", out)

scale = 1 - SUBNET_OWNER_WEIGHT_SHARE
assert_close(out[0], 0.2 * scale + SUBNET_OWNER_WEIGHT_SHARE, msg="owner = old*(1-share) + share")
assert_close(out[1], 0.8 * scale, msg="miner-1 scaled")
assert_close(sum(out.values()), 1.0, msg="sum = 1.0")

In [ ]:
# share=0 -> passthrough.
print("share=0 (passthrough)")
inp = {1: 0.6, 2: 0.4}
out = reserve_subnet_owner_share(inp, share=0.0)
show("output", out)
assert out == inp and SUBNET_OWNER_UID not in out
print("  [PASS] share=0 returns the input unchanged")

In [ ]:
# share=1 -> everything collapses to the owner.
print("share=1 (collapse to owner)")
out = reserve_subnet_owner_share({1: 0.5, 2: 0.5}, share=1.0)
show("output", out)
assert out == {SUBNET_OWNER_UID: 1.0}
print("  [PASS] share=1 -> {0: 1.0}")

In [ ]:
# Custom owner UID (defaults to SUBNET_OWNER_UID=0; can override).
print("custom owner uid=42")
out = reserve_subnet_owner_share({1: 1.0}, owner_uid=42, share=0.1)
show("output", out)
assert_close(out[42], 0.1, msg="custom owner gets share")
assert_close(out[1], 0.9, msg="miner scaled by 0.9")

## 4. Full submission pipeline

What actually hits the chain. The validator path is:

1. `build_submission_uid_weights` -> `{uid: weight}` (sums to ~1.0)
2. `reserve_subnet_owner_share` -> `SUBNET_OWNER_WEIGHT_SHARE` reserved for uid=0
3. `_normalize_uid_weights` (drops non-positive, applies `top_k`, normalizes) -> `(uids, weights)` passed to `set_weights`

With `top_k=None` (the production default when `enable_round_group_construction=True`), step 3 is a no-op on the reserved share — uid=0 ends up at exactly `SUBNET_OWNER_WEIGHT_SHARE` on chain.

In [ ]:
raw = {12: 0.441, 27: 0.323, 41: 0.196, 88: 0.025, 91: 0.015}
reserved = reserve_subnet_owner_share(raw)
uids, weights = _normalize_uid_weights(reserved, normalize=True, top_k=None)
submitted = dict(zip(uids, weights))

show("raw       ", raw)
show("+ reserve ", reserved)
show("on chain  ", submitted)

assert_close(submitted[SUBNET_OWNER_UID], SUBNET_OWNER_WEIGHT_SHARE, msg="on-chain owner share == SUBNET_OWNER_WEIGHT_SHARE")
assert_close(sum(submitted.values()), 1.0, msg="on-chain weights sum to 1.0")

## 5. The `top_k` caveat

The helper's docstring warns: if downstream `top_k` filtering is tight enough to drop uid=0's reserved slice, the reserve is lost. The production `ChainSubmitter` is instantiated with `top_k=None` whenever `enable_round_group_construction=True`, so this is safe in normal operation. But if you flip back to the legacy path (`enable_round_group_construction=False`) with `top_k_miners_to_reward=3`, the smallest entry — which (for typical payloads) is uid=0's `SUBNET_OWNER_WEIGHT_SHARE` slice — gets cut.

Reproduce that failure mode below so the boundary is visible.

In [ ]:
reserved = reserve_subnet_owner_share({12: 0.441, 27: 0.323, 41: 0.196, 88: 0.025, 91: 0.015})
show("reserved   (before top_k)", reserved)

for k in (3, 4, 5, 6):
    uids, weights = _normalize_uid_weights(reserved, normalize=True, top_k=k)
    sub = dict(zip(uids, weights))
    has_owner = SUBNET_OWNER_UID in sub
    owner_w = sub.get(SUBNET_OWNER_UID, 0.0)
    print(f"\ntop_k={k} -> uids={sorted(sub)}")
    print(f"  owner present? {has_owner}    owner weight: {owner_w:.4f}")

print("\nTakeaway: uid=0's 5% slice is the smallest entry, so a tight top_k drops it.")
print("Production avoids this by setting top_k=None when round-group construction is on.")

## 6. Fallback path mirror

The peer-consensus fallback emits even weight across the top-3 differentiated miners. After the reserve, each miner gets `(1 - SUBNET_OWNER_WEIGHT_SHARE) / 3`, and uid=0 gets `SUBNET_OWNER_WEIGHT_SHARE`.

In [ ]:
top3 = [12, 27, 41]
miner_weights_raw = {u: 1.0 / len(top3) for u in top3}
out = reserve_subnet_owner_share(miner_weights_raw)

show("raw even-3", miner_weights_raw)
show("+ reserve ", out)

for u in top3:
    assert_close(out[u], (1 - SUBNET_OWNER_WEIGHT_SHARE) / 3, msg=f"uid={u} == (1-share)/3")
assert_close(out[SUBNET_OWNER_UID], SUBNET_OWNER_WEIGHT_SHARE, msg="owner == SUBNET_OWNER_WEIGHT_SHARE")
assert_close(sum(out.values()), 1.0, msg="sum = 1.0")

## Summary

* Normal `{uid: weight}` payloads -> uid=0 always lands at exactly `SUBNET_OWNER_WEIGHT_SHARE` on chain when downstream `top_k=None`.
* Empty / collapsing inputs -> uid=0 gets 100%.
* Owner already in payload -> stacks: `owner * (1 - share) + share`.
* Tight `top_k` (<= number of miners) -> reserved slice gets cut; avoid in production.